In [ ]:
import os
import re
import json
import logging
import fitz  # PyMuPDF
import docx
import easyocr
import numpy as np
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment


In [ ]:
BASE_DIR = Path().resolve().parents[2]
DATA_DIR = BASE_DIR / "Переводы" / "Переводы"
OUTPUT_JSONL = Path().resolve() / "dataset_full_docs.jsonl"
LOG_FILE = Path().resolve() / "res.log"

In [ ]:
logger = logging.getLogger("DatasetParser")
logger.setLevel(logging.INFO)
if logger.hasHandlers():
    logger.handlers.clear()


console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))
logger.addHandler(console_handler)

file_handler = logging.FileHandler(LOG_FILE, mode='w', encoding='utf-8')
file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
logger.addHandler(file_handler)

logger.info(f"Директория с данными: {DATA_DIR}")
logger.info(f"Файл вывода: {OUTPUT_JSONL}")

In [ ]:
logger.info("Загрузка модели EasyOCR")
reader = easyocr.Reader(['ru', 'en'], gpu=False)
logger.info("Модель загружена")

In [ ]:
def extract_from_docx(path: Path) -> str:
    """Извлекает текст из абзацев и всех таблиц (включая вложенные)"""
    try:
        doc = docx.Document(path)
        texts = []

        # 1. Извлекаем обычные абзацы
        for p in doc.paragraphs:
            if p.text.strip():
                texts.append(p.text.strip())

        # 2. Рекурсивная функция извлечения из таблиц
        def extract_table(table):
            for row in table.rows:
                for cell in row.cells:
                    # Текст внутри ячейки
                    for p in cell.paragraphs:
                        if p.text.strip():
                            texts.append(p.text.strip())
                    # Вложенные таблицы
                    for nested_table in cell.tables:
                        extract_table(nested_table)

        # Запускаем сбор из таблиц
        for table in doc.tables:
            extract_table(table)

        return "\n".join(texts)
    except Exception as e:
        logger.error(f"Ошибка при чтении DOCX {path.name}: {e}")
        return ""

def extract_from_pdf_normal(path: Path) -> str:
    try:
        doc = fitz.open(path)
        text = ""
        for page in doc:
            text += page.get_text() + "\n"
        doc.close()
        return text.strip()
    except Exception as e:
        logger.error(f"Ошибка при стандартном чтении PDF {path.name}: {e}")
        return ""

def extract_from_pdf_ocr(path: Path) -> str:
    try:
        doc = fitz.open(path)
        text = ""
        for page in doc:
            mat = fitz.Matrix(2.0, 2.0)
            pix = page.get_pixmap(matrix=mat)
            img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)

            results = reader.readtext(img_array, detail=0)
            text += " ".join(results) + "\n"
        doc.close()
        return text.strip()
    except Exception as e:
        logger.error(f"Ошибка при OCR чтении PDF {path.name}: {e}")
        return ""

def extract_text(path: Path) -> str:
    ext = path.suffix.lower()
    text = ""

    if ext == ".docx":
        text = extract_from_docx(path)

    elif ext == ".pdf":
        text = extract_from_pdf_normal(path)
        if len(text.strip()) < 20:
            logger.warning(f"PDF не содержит текстового слоя. Подключаем OCR: {path.name}")
            text = extract_from_pdf_ocr(path)
            if not text:
                logger.warning(f"OCR также не смог распознать текст в: {path.name}")
    else:
        logger.warning(f"Неизвестный формат файла: {path.name}")

    return text.strip()

In [ ]:
with open(OUTPUT_JSONL, 'w', encoding='utf-8') as jsonl_file:
    for domain_dir in [d for d in DATA_DIR.iterdir() if d.is_dir()]:
        domain = domain_dir.name
        logger.info(f"--- Обработка тематики: {domain} ---")

        pairs = {}
        for file_path in domain_dir.iterdir():
            if file_path.is_file():
                match = re.search(r'(ENG|RUS)(\d+)', file_path.stem, re.IGNORECASE)
                if match:
                    lang_prefix = match.group(1).upper()
                    doc_id = match.group(2)
                    lang_key = "en" if lang_prefix == "ENG" else "ru"

                    if doc_id not in pairs:
                        pairs[doc_id] = {}
                    pairs[doc_id][lang_key] = file_path

        for doc_id, files in pairs.items():
            if "en" in files and "ru" in files:
                en_path = files["en"]
                ru_path = files["ru"]

                logger.info(f"Парсинг пары #{doc_id}: {en_path.name} <-> {ru_path.name}")

                en_text = extract_text(en_path)
                ru_text = extract_text(ru_path)

                len_en = len(en_text)
                len_ru = len(ru_text)

                if len_en == 0 or len_ru == 0:
                    logger.error(f"Пара #{doc_id} пропущена: один из файлов пуст.")
                    continue

                max_len = max(len_en, len_ru)
                diff_percent = (abs(len_en - len_ru) / max_len) * 100
                row = {
                    "en": en_text,
                    "ru": ru_text,
                    "domain": domain,
                    "doc_name": en_path.stem
                }
                jsonl_file.write(json.dumps(row, ensure_ascii=False) + "\n")
                jsonl_file.flush()


In [ ]:
INPUT_JSONL = Path().resolve() / "dataset_full_docs.jsonl"
OUTPUT_JSONL = Path().resolve() / "dataset_nllb_semantic.jsonl"
LOG_FILE = Path().resolve() / "res_alignment.log"

logger = logging.getLogger("Semantic_Aligner")
logger.setLevel(logging.INFO)
if logger.hasHandlers(): logger.handlers.clear()
logger.addHandler(logging.StreamHandler())

logger.info("Загрузка модели эмбеддингов...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
logger.info("Модель загружена!")

In [ ]:
def clean_and_split(text: str) -> list[str]:
    """Разбивает текст на куски, убирая совсем пустые строки"""
    lines = text.split('\n')
    clean_lines = [line.strip() for line in lines if len(line.strip()) > 3]
    return clean_lines

def semantic_align(en_text: str, ru_text: str, threshold: float = 0.65) -> list[tuple]:
    """
    Выравнивание на основе матрицы косинусного сходства и венгерского алгоритма.
    threshold - минимальный порог уверенности (от 0 до 1).
    """
    en_lines = clean_and_split(en_text)
    ru_lines = clean_and_split(ru_text)

    if not en_lines or not ru_lines:
        return []

    # 1. Получаем векторы для всех строк
    en_embeddings = model.encode(en_lines)
    ru_embeddings = model.encode(ru_lines)

    # 2. Строим матрицу смежности (Cosine Similarity Matrix)
    # Размерность: [len(en_lines), len(ru_lines)]
    sim_matrix = cosine_similarity(en_embeddings, ru_embeddings)

    # 3. Венгерский алгоритм (ищет максимальное совпадение, игнорируя порядок!)
    # Функция ищет минимум, поэтому передаем отрицательную матрицу
    row_ind, col_ind = linear_sum_assignment(-sim_matrix)

    chunks = []
    # 4. Проходим по результатам алгоритма и фильтруем по порогу уверенности
    for i, j in zip(row_ind, col_ind):
        score = sim_matrix[i, j]

        # Если смысл предложений совпадает больше чем на threshold (65%)
        if score >= threshold:

            chunks.append((en_lines[i], ru_lines[j]))

    return chunks


docs, good_chunks = 0, 0
with open(INPUT_JSONL, 'r', encoding='utf-8') as f_in, open(OUTPUT_JSONL, 'w', encoding='utf-8') as f_out:
    for line in f_in:
        if not line.strip(): continue
        data = json.loads(line)
        docs += 1

        chunks = semantic_align(data.get("en", ""), data.get("ru", ""), threshold=0.65)

        for i, (e, r) in enumerate(chunks):
            row = {"en": e, "ru": r, "domain": data.get("domain", ""), "doc_name": data.get("doc_name", "")}
            f_out.write(json.dumps(row, ensure_ascii=False) + "\n")
            good_chunks += 1

logger.info(f"Обраработано документов: {docs}")
logger.info(f"Семантических пар получено: {good_chunks}")

In [ ]:
def clean_and_split(text: str) -> list[str]:
    lines = text.split('\n')
    return [line.strip() for line in lines if len(line.strip()) > 3]

def refine_pair(en_txt: str, ru_txt: str) -> tuple:
    """
    Пост-обработка: отрезает мусор от длинной строки, сверяя куски через нейросеть.
    Возвращает (очищенный_en, очищенный_ru) или (None, None), если это брак.
    """
    len_e = max(1, len(en_txt))
    len_r = max(1, len(ru_txt))
    ratio = len_r / len_e

    if 0.8 <= ratio <= 1.35:
        return en_txt, ru_txt

    # Определяем, кто длиннее
    is_ru_longer = ratio > 1.35
    long_txt = ru_txt if is_ru_longer else en_txt
    short_txt = en_txt if is_ru_longer else ru_txt

    # 1. Пытаемся разбить длинный текст по знакам препинания
    parts = [p.strip() for p in re.split(r'[,:;]\s*|\s+[-—]\s+', long_txt) if p.strip()]

    # 2. Если знаков препинания нет, бьем пополам (по пробелу)
    if len(parts) < 2:
        words = long_txt.split()
        if len(words) >= 4: # Бьем только если слов хотя бы 4
            mid = len(words) // 2
            parts = [" ".join(words[:mid]), " ".join(words[mid:])]
        else:
            # Если слов мало, а разница длин огромная - это мусор, бракуем
            if ratio > 2.5 or ratio < 0.4:
                return None, None
            return en_txt, ru_txt

    # Получаем векторы для короткого текста и всех наших "кусочков"
    short_emb = model.encode([short_txt])
    parts_emb = model.encode(parts)

    # Сравниваем кусочки с коротким текстом
    sims = cosine_similarity(short_emb, parts_emb)[0]
    best_idx = np.argmax(sims)
    best_score = sims[best_idx]
    best_part = parts[best_idx]

    # Если лучший кусочек сохранил смысл (score > 0.55), берем его!
    if best_score >= 0.55:
        if is_ru_longer:
            return short_txt, best_part
        else:
            return best_part, short_txt

    # Если ни один кусок не подошел, бракуем при сильном рассинхроне
    if ratio > 2.0 or ratio < 0.5:
        return None, None

    return en_txt, ru_txt

In [ ]:
def semantic_align(en_text: str, ru_text: str, threshold: float = 0.65) -> list[tuple]:
    en_lines = clean_and_split(en_text)
    ru_lines = clean_and_split(ru_text)

    if not en_lines or not ru_lines:
        return []

    en_embeddings = model.encode(en_lines)
    ru_embeddings = model.encode(ru_lines)

    sim_matrix = cosine_similarity(en_embeddings, ru_embeddings)
    row_ind, col_ind = linear_sum_assignment(-sim_matrix)

    chunks = []
    refined_count = 0

    for i, j in zip(row_ind, col_ind):
        score = sim_matrix[i, j]

        if score >= threshold:

            final_en, final_ru = refine_pair(en_lines[i], ru_lines[j])

            if final_en and final_ru:
                # Считаем статистику улучшений
                if final_en != en_lines[i] or final_ru != ru_lines[j]:
                    refined_count += 1
                chunks.append((final_en, final_ru))

    return chunks, refined_count


docs, good_chunks, total_refined = 0, 0, 0
with open(INPUT_JSONL, 'r', encoding='utf-8') as f_in, open(OUTPUT_JSONL, 'w', encoding='utf-8') as f_out:
    for line in f_in:
        if not line.strip(): continue
        data = json.loads(line)
        docs += 1

        chunks, refined = semantic_align(data.get("en", ""), data.get("ru", ""), threshold=0.65)
        total_refined += refined

        for i, (e, r) in enumerate(chunks):
            row = {"en": e, "ru": r, "domain": data.get("domain", ""), "doc_name": data.get("doc_name_en", "")}
            f_out.write(json.dumps(row, ensure_ascii=False) + "\n")
            good_chunks += 1

logger.info(f"Обраработано документов: {docs}")
logger.info(f"Cемантических пар получено: {good_chunks}")
logger.info(f"Исправлено/отрезано мусора (благодаря пост-обработке): {total_refined} пар!")

main()

In [ ]:
def basic_split(text: str) -> list[str]:
    """Агрессивная разбивка: лечит слипшиеся слова и убивает технический мусор"""

    # 1. Вырезаем типичный непереводимый мусор
    text = re.sub(r'(?i)(TEL|FAX|Email|Ph|Mob)[\s:.-]*[\w@.-]+', '', text)

    # 2. Лечим слипшиеся слова с цифрами
    text = re.sub(r'([a-zа-я])(\d+\.)', r'\1 \2', text)

    # 3. Лечим слипшуюся пунктуацию
    text = re.sub(r'([.!?…;:])([A-ZА-ЯЁ])', r'\1 \2', text)

    # 4. Слипшаяся строчная и Заглавная
    text = re.sub(r'([a-zа-яё])([A-ZА-ЯЁ])', r'\1. \2', text)

    # 5. Строчная, много пробелов, Заглавная
    text = re.sub(r'([a-zа-яё])\s{2,}([A-ZА-ЯЁ])', r'\1. \2', text)

    # 6. Бьем текст по знакам препинания и оставшимся (законным) переносам строк
    raw = re.split(r'\n|(?<=[.!?…;:;])\s+', text)

    # 7. Фильтруем мусор
    return [s.strip() for s in raw if len(re.findall(r'[a-zA-Zа-яА-ЯёЁ]', s)) >= 3]

def stage1_semantic_filter(en_text: str, ru_text: str, threshold: float = 0.65) -> list[tuple]:
    en_lines = basic_split(en_text)
    ru_lines = basic_split(ru_text)

    if not en_lines or not ru_lines:
        return []

    en_emb = model.encode(en_lines)
    ru_emb = model.encode(ru_lines)

    sim_matrix = cosine_similarity(en_emb, ru_emb)
    row_ind, col_ind = linear_sum_assignment(-sim_matrix)

    matched_pairs = []

    for i, j in zip(row_ind, col_ind):
        if sim_matrix[i, j] >= threshold:
            e_str = en_lines[i]
            r_str = ru_lines[j]

            # --- НОВАЯ ЗАЩИТА: ПРОВЕРКА ДЛИНЫ ---
            len_e = max(1, len(e_str))
            len_r = max(1, len(r_str))
            ratio = len_r / len_e

            # Если один текст длиннее другого более чем в 2.2 раза - это БРАК, выкидываем!
            if ratio < 0.45 or ratio > 2.2:
                continue

            en_nums = set(re.findall(r'\d+', e_str))
            ru_nums = set(re.findall(r'\d+', r_str))
            if en_nums and ru_nums and not en_nums.intersection(ru_nums):
                continue

            matched_pairs.append((i, e_str, r_str))

    matched_pairs.sort(key=lambda x: x[0])
    return [(pair[1], pair[2]) for pair in matched_pairs]

def stage2_chunk_for_nllb(perfect_pairs: list[tuple], target_chars: int = 250) -> list[tuple]:
    """ЭТАП 2: Берем идеальные пары и склеиваем их в чанки нужного размера"""
    final_chunks = []
    curr_en, curr_ru = "", ""

    for en_sent, ru_sent in perfect_pairs:
        # Если при добавлении следующего предложения мы превысим лимит,
        # то сохраняем текущий кусок и начинаем новый
        if len(curr_en) + len(en_sent) > target_chars:
            if curr_en and curr_ru:
                final_chunks.append((curr_en.strip(), curr_ru.strip()))
            curr_en = en_sent
            curr_ru = ru_sent
        else:
            # Иначе приклеиваем предложение
            curr_en += " " + en_sent if curr_en else en_sent
            curr_ru += " " + ru_sent if curr_ru else ru_sent

    if curr_en and curr_ru:
         final_chunks.append((curr_en.strip(), curr_ru.strip()))

    return final_chunks


docs, good_chunks = 0, 0
with open(INPUT_JSONL, 'r', encoding='utf-8') as f_in, open(OUTPUT_JSONL, 'w', encoding='utf-8') as f_out:
    for line in f_in:
        if not line.strip(): continue
        data = json.loads(line)
        docs += 1

        # ЭТАП 1: Вытаскиваем только то, в чем уверены
        perfect_pairs = stage1_semantic_filter(data.get("en", ""), data.get("ru", ""), threshold=0.75)

        # ЭТАП 2: Упаковываем это в красивые чанки для NLLB
        nllb_chunks = stage2_chunk_for_nllb(perfect_pairs, target_chars=200)

        for i, (e, r) in enumerate(nllb_chunks):
            row = {"en": e, "ru": r, "domain": data.get("domain", ""), "doc_name": data.get("doc_name_en", "")}
            f_out.write(json.dumps(row, ensure_ascii=False) + "\n")
            good_chunks += 1

logger.info(f"Обраработано документов: {docs}")
logger.info(f"Итого идеальных чанков для обучения NLLB: {good_chunks}")

In [ ]:
def generate_candidates(sentences: list[str], max_merge: int = 3) -> list[tuple]:
    """
    Генерирует кандидатов для выравнивания: объединяет соседние предложения.
    Возвращает список кортежей: (start_idx, end_idx, merged_text)
    """
    candidates = []
    n = len(sentences)

    for start in range(n):
        for end in range(start, min(n, start + max_merge)):
            merged = " ".join(sentences[start:end+1]).strip()
            if merged:
                candidates.append((start, end, merged))

    return candidates

def stage1_extended_filter(en_text: str, ru_text: str, threshold: float = 0.75, max_merge: int = 3) -> list[tuple]:
    """
    Этап 1: Расширенное семантическое выравнивание с возможностью объединения соседних строк.
    Возвращает список кортежей: (en_text, ru_text, confidence, len_diff)
    """
    # Разбиваем на базовые строки
    en_sents = basic_split(en_text)
    ru_sents = basic_split(ru_text)

    if not en_sents or not ru_sents:
        logger.debug(f"Пустой текст после разбивки: EN={len(en_sents)}, RU={len(ru_sents)}")
        return []

    # Генерируем кандидатов (объединения соседних строк)
    en_candidates = generate_candidates(en_sents, max_merge)
    ru_candidates = generate_candidates(ru_sents, max_merge)

    # Извлекаем тексты кандидатов для эмбеддинга
    en_texts = [txt for _, _, txt in en_candidates]
    ru_texts = [txt for _, _, txt in ru_candidates]

    # Получаем эмбеддинги
    try:
        en_embeddings = model.encode(en_texts, show_progress_bar=False)
        ru_embeddings = model.encode(ru_texts, show_progress_bar=False)
    except Exception as e:
        logger.error(f"Ошибка при кодировании: {e}")
        return []

    # Вычисляем матрицу схожести
    similarity_matrix = cosine_similarity(en_embeddings, ru_embeddings)

    # Собираем все потенциальные пары выше порога
    candidate_pairs = []
    for i in range(len(en_candidates)):
        for j in range(len(ru_candidates)):
            sim = similarity_matrix[i, j]
            if sim >= threshold:
                candidate_pairs.append((sim, i, j))

    # Сортируем по убыванию уверенности (чтобы лучшие пары обрабатывались первыми)
    candidate_pairs.sort(key=lambda x: x[0], reverse=True)

    # Жадно выбираем непересекающиеся пары
    used_en_indices = set()  # Индексы базовых предложений EN, которые уже использованы
    used_ru_indices = set()  # Индексы базовых предложений RU, которые уже использованы
    aligned_pairs = []

    for confidence, en_idx, ru_idx in candidate_pairs:
        en_start, en_end, en_str = en_candidates[en_idx]
        ru_start, ru_end, ru_str = ru_candidates[ru_idx]

        # Проверка на количество букв (минимум 3 буквы в каждом)
        if len(re.findall(r'[a-zA-Zа-яА-ЯёЁ]', en_str)) < 3 or len(re.findall(r'[a-zA-Zа-яА-ЯёЁ]', ru_str)) < 3:
            continue

        # Проверка на совпадение чисел
        en_nums = set(re.findall(r'\d+', en_str))
        ru_nums = set(re.findall(r'\d+', ru_str))
        if en_nums and ru_nums and not en_nums.intersection(ru_nums):
            logger.debug(f"Пропущена пара из-за несовпадения чисел: '{en_str[:50]}...' <-> '{ru_str[:50]}...'")
            continue

        # Проверка соотношения длин
        len_en = max(1, len(en_str))
        len_ru = max(1, len(ru_str))
        ratio = len_ru / len_en

        if ratio < 0.45 or ratio > 2.2:
            logger.debug(f"Пропущена пара из-за соотношения длин ({ratio:.2f}): '{en_str[:50]}...' <-> '{ru_str[:50]}...'")
            continue

        # Проверяем, что эти базовые предложения еще не использованы
        en_range = set(range(en_start, en_end + 1))
        ru_range = set(range(ru_start, ru_end + 1))

        if not (en_range & used_en_indices) and not (ru_range & used_ru_indices):
            # Пара принимается
            len_diff = abs(len_en - len_ru)
            aligned_pairs.append((en_start, en_str, ru_str, confidence, len_diff))

            # Помечаем использованные индексы
            used_en_indices.update(en_range)
            used_ru_indices.update(ru_range)

            logger.debug(f"Выровнена пара (conf={confidence:.3f}): '{en_str[:50]}...' <-> '{ru_str[:50]}...'")

    # Сортируем по порядку в исходном английском тексте
    aligned_pairs.sort(key=lambda x: x[0])

    # Возвращаем без индекса, только тексты и метрики
    return [(en, ru, conf, diff) for _, en, ru, conf, diff in aligned_pairs]

def stage2_chunk_for_nllb(perfect_pairs: list[tuple], target_chars: int = 200) -> list[tuple]:
    """
    Этап 2: Берем идеальные пары и склеиваем их в чанки нужного размера.
    Принимает список кортежей (en, ru) и возвращает чанки (en, ru)
    """
    if not perfect_pairs:
        return []

    final_chunks = []
    curr_en, curr_ru = "", ""

    for pair in perfect_pairs:
        en_sent, ru_sent = pair[0], pair[1]  # Берем только тексты, игнорируем остальные поля

        # Если при добавлении следующего предложения мы превысим лимит,
        # то сохраняем текущий кусок и начинаем новый
        if curr_en and len(curr_en) + len(en_sent) + 1 > target_chars:  # +1 для пробела
            if curr_en.strip() and curr_ru.strip():
                final_chunks.append((curr_en.strip(), curr_ru.strip()))
                logger.debug(f"Создан чанк: EN={len(curr_en)} симв, RU={len(curr_ru)} симв")
            curr_en = en_sent
            curr_ru = ru_sent
        else:
            # Приклеиваем предложение
            if curr_en:
                curr_en += " " + en_sent
                curr_ru += " " + ru_sent
            else:
                curr_en = en_sent
                curr_ru = ru_sent

    # Сохраняем остатки
    if curr_en.strip() and curr_ru.strip():
        final_chunks.append((curr_en.strip(), curr_ru.strip()))
        logger.debug(f"Финальный чанк: EN={len(curr_en)} симв, RU={len(curr_ru)} симв")

    return final_chunks

def save_logs(confidence_log: list, length_log: list):
    """Сохраняет логи уверенности и разницы длин в JSONL файлы"""

    # Лог уверенности (сортировка от неуверенного к уверенному)
    if confidence_log:
        confidence_log.sort(key=lambda x: x[2])  # Сортировка по confidence (возрастание)

        with open(CONFIDENCE_LOG, 'w', encoding='utf-8') as f:
            for en, ru, conf in confidence_log:
                record = {
                    "en": en,
                    "ru": ru,
                    "confidence": round(float(conf), 4)
                }
                f.write(json.dumps(record, ensure_ascii=False) + '\n')

        logger.info(f"Сохранен лог уверенности: {len(confidence_log)} записей в {CONFIDENCE_LOG}")

    # Лог разницы длин (сортировка от наибольшей разницы к наименьшей)
    if length_log:
        length_log.sort(key=lambda x: x[2], reverse=True)  # Сортировка по len_diff (убывание)

        with open(LENGTH_DIFF_LOG, 'w', encoding='utf-8') as f:
            for en, ru, diff in length_log:
                record = {
                    "en": en,
                    "ru": ru,
                    "len_diff": diff,
                    "en_len": len(en),
                    "ru_len": len(ru)
                }
                f.write(json.dumps(record, ensure_ascii=False) + '\n')

        logger.info(f"Сохранен лог разницы длин: {len(length_log)} записей в {LENGTH_DIFF_LOG}")


logger.info(f"Начало обработки файла: {INPUT_JSONL}")
logger.info(f"Результаты будут сохранены в: {OUTPUT_JSONL}")

# Статистика
docs_processed = 0
docs_with_pairs = 0
total_chunks = 0
total_pairs = 0

# Для логов
all_confidence_log = []
all_length_log = []

try:
    with open(INPUT_JSONL, 'r', encoding='utf-8') as f_in, \
         open(OUTPUT_JSONL, 'w', encoding='utf-8') as f_out:

        for line_num, line in enumerate(f_in, 1):
            if not line.strip():
                continue

            try:
                data = json.loads(line)
            except json.JSONDecodeError as e:
                logger.warning(f"Строка {line_num}: ошибка парсинга JSON - {e}")
                continue

            docs_processed += 1

            # Получаем тексты
            en_text = data.get("en", "")
            ru_text = data.get("ru", "")

            if not en_text or not ru_text:
                logger.debug(f"Документ {docs_processed}: пустой EN или RU текст")
                continue

            # Этап 1: Расширенное семантическое выравнивание
            perfect_pairs = stage1_extended_filter(
                en_text,
                ru_text,
                threshold=0.75,  # Порог уверенности
                max_merge=2       # Максимальное количество объединяемых предложений
            )

            if not perfect_pairs:
                logger.debug(f"Документ {docs_processed}: не найдено пар после фильтрации")
                continue

            docs_with_pairs += 1
            total_pairs += len(perfect_pairs)

            # Собираем данные для логов
            for en, ru, conf, diff in perfect_pairs:
                all_confidence_log.append((en, ru, conf))
                all_length_log.append((en, ru, diff))

            # Этап 2: Создание чанков для NLLB
            pairs_for_chunking = [(en, ru) for en, ru, _, _ in perfect_pairs]
            nllb_chunks = stage2_chunk_for_nllb(pairs_for_chunking, target_chars=200)

            # Сохраняем чанки
            domain = data.get("domain", "")
            doc_name = data.get("doc_name_en", data.get("doc_name", ""))

            for chunk_en, chunk_ru in nllb_chunks:
                row = {
                    "en": chunk_en,
                    "ru": chunk_ru,
                    "domain": domain,
                    "doc_name": doc_name
                }
                f_out.write(json.dumps(row, ensure_ascii=False) + '\n')
                total_chunks += 1

            # Прогресс каждые 100 документов
            if docs_processed % 100 == 0:
                logger.info(f"Обработано документов: {docs_processed}, "
                          f"найдено пар: {total_pairs}, "
                          f"создано чанков: {total_chunks}")

    logger.info(f"Обработка завершена!")
    logger.info(f"Всего обработано документов: {docs_processed}")
    logger.info(f"Документов с найденными парами: {docs_with_pairs}")
    logger.info(f"Всего найдено пар предложений: {total_pairs}")
    logger.info(f"Создано чанков для NLLB: {total_chunks}")

    # Сохраняем логи
    save_logs(all_confidence_log, all_length_log)

    # Дополнительная статистика
    if all_confidence_log:
        confidences = [conf for _, _, conf in all_confidence_log]
        logger.info(f"Статистика уверенности: min={min(confidences):.4f}, "
                  f"max={max(confidences):.4f}, "
                  f"avg={np.mean(confidences):.4f}")

    if all_length_log:
        diffs = [diff for _, _, diff in all_length_log]
        logger.info(f"Статистика разницы длин: min={min(diffs)}, "
                  f"max={max(diffs)}, "
                  f"avg={np.mean(diffs):.1f}")

except FileNotFoundError:
    logger.error(f"Входной файл не найден: {INPUT_JSONL}")
except Exception as e:
    logger.error(f"Неожиданная ошибка: {e}", exc_info=True)

In [ ]:
def basic_split(text: str, split_commas: bool = False) -> list[tuple[int, int, str]]:
    """
    Разбивает текст на строки. Возвращает список (start, end, text) в координатах исходного текста.
    Если split_commas=True, дополнительно разбивает по запятым (с ограничениями).
    """
    # Сначала стандартная очистка и грубое деление
    cleaned = text
    cleaned = re.sub(r'(?i)(TEL|FAX|Email|Ph|Mob)[\s:.-]*[\w@.-]+', '', cleaned)
    cleaned = re.sub(r'([a-zа-я])(\d+\.)', r'\1 \2', cleaned)
    cleaned = re.sub(r'([.!?…;:])([A-ZА-ЯЁ])', r'\1 \2', cleaned)
    cleaned = re.sub(r'([a-zа-яё])([A-ZА-ЯЁ])', r'\1. \2', cleaned)
    cleaned = re.sub(r'([a-zа-яё])\s{2,}([A-ZА-ЯЁ])', r'\1. \2', cleaned)

    raw_parts = re.split(r'\n|(?<=[.!?…;:;])\s+', cleaned)

    segments = []  # будет список (start, end, text)
    pos = 0
    for part in raw_parts:
        part = part.strip()
        if not part:
            continue
        # Ищем эту часть в оригинальном тексте начиная с pos
        idx = cleaned.find(part, pos)
        if idx == -1:
            # Если не нашли (например, из-за регулярки), помечаем приблизительно
            start = pos
            end = pos + len(part)
            pos = end
        else:
            start = idx
            end = idx + len(part)
            pos = end
        if len(re.findall(r'[a-zA-Zа-яА-ЯёЁ]', part)) >= 3:
            segments.append((start, end, part))

    if not split_commas:
        return segments

    # Дополнительное дробление по запятым с эвристиками
    new_segments = []
    for start, end, s in segments:
        # Разбиваем по запятой с пробелом, но не после цифры и не перед цифрой
        sub_parts = re.split(r'(?<!\d),(?=\s*[^\d])', s)
        # Если получилось больше одной части – обрабатываем
        if len(sub_parts) > 1:
            sub_start = start
            for sub in sub_parts:
                sub = sub.strip()
                if not sub:
                    continue
                sub_end = sub_start + len(sub)
                if len(re.findall(r'[a-zA-Zа-яА-ЯёЁ]', sub)) >= 3:
                    new_segments.append((sub_start, sub_end, sub))
                sub_start = sub_end + 1  # примерно учитываем запятую и пробел
        else:
            new_segments.append((start, end, s))
    return new_segments

def generate_candidates(segments: list[tuple[int, int, str]], max_merge: int = 3,
                        enable_broken_merge: bool = True, max_broken_merge: int = 5) -> list[tuple]:
    """
    Генерирует кандидатов для выравнивания:
    - стандартные комбинации в окне max_merge;
    - дополнительные комбинации для «разорванных» строк (если нет финального знака препинания).
    Возвращает список (start, end, merged_text, seg_count).
    """
    n = len(segments)
    candidates_dict = {}  # ключ (start, end) для уникальности

    # 1. Стандартные комбинации (как раньше)
    for i in range(n):
        for j in range(i, min(n, i + max_merge)):
            start = segments[i][0]
            end = segments[j][1]
            merged = " ".join(segments[k][2] for k in range(i, j+1))
            seg_count = j - i + 1
            candidates_dict[(start, end)] = (start, end, merged, seg_count)

    if not enable_broken_merge:
        return list(candidates_dict.values())

    # 2. Эвристика «разорванных строк»
    # Ищем цепочки, где каждый сегмент (кроме, возможно, последнего) не заканчивается знаком конца предложения
    i = 0
    while i < n:
        # Если текущий сегмент не заканчивается на .!?…;:, начинаем цепочку
        if not re.search(r'[.!?…;:]\s*$', segments[i][2]):
            chain_start = i
            j = i + 1
            while j < n and (j - chain_start) < max_broken_merge:
                # Добавляем в кандидаты цепочку от chain_start до j (включительно)
                start = segments[chain_start][0]
                end = segments[j][1]
                merged = " ".join(segments[k][2] for k in range(chain_start, j+1))
                seg_count = j - chain_start + 1
                # Сохраняем, если такого ещё нет
                if (start, end) not in candidates_dict:
                    candidates_dict[(start, end)] = (start, end, merged, seg_count)
                # Если j-й сегмент заканчивается знаком — цепочка завершается
                if re.search(r'[.!?…;:]\s*$', segments[j][2]):
                    break
                j += 1
            i = j if j < n and re.search(r'[.!?…;:]\s*$', segments[j-1][2]) else i+1
        else:
            i += 1

    return list(candidates_dict.values())

def stage1_extended_filter(en_text: str, ru_text: str,
                           threshold: float = 0.75, max_merge: int = 2,
                           split_commas: bool = False) -> list[tuple]:
    """
    Выравнивание с поддержкой объединения строк и опциональным split_commas.
    Возвращает список: (en_start, en_end, ru_start, ru_end, en_str, ru_str, confidence, len_diff, total_seg_count)
    """
    en_segments = basic_split(en_text, split_commas=split_commas)
    ru_segments = basic_split(ru_text, split_commas=split_commas)

    if not en_segments or not ru_segments:
        return []

    en_candidates = generate_candidates(en_segments, max_merge, enable_broken_merge=True)
    ru_candidates = generate_candidates(ru_segments, max_merge, enable_broken_merge=True)

    en_texts = [txt for _, _, txt, _ in en_candidates]
    ru_texts = [txt for _, _, txt, _ in ru_candidates]

    try:
        en_emb = model.encode(en_texts, show_progress_bar=False)
        ru_emb = model.encode(ru_texts, show_progress_bar=False)
    except Exception as e:
        logger.error(f"Ошибка кодирования: {e}")
        return []

    sim = cosine_similarity(en_emb, ru_emb)

    # Собираем пары выше порога
    cand_pairs = []
    for i in range(len(en_candidates)):
        for j in range(len(ru_candidates)):
            score = sim[i, j]
            if score >= threshold:
                en_seg_count = en_candidates[i][3]
                ru_seg_count = ru_candidates[j][3]
                total_seg = en_seg_count + ru_seg_count
                cand_pairs.append((score, total_seg, i, j))

    # Сортировка: сначала по убыванию score, затем по возрастанию суммарного количества сегментов
    cand_pairs.sort(key=lambda x: (-x[0], x[1]))

    used_en = set()
    used_ru = set()
    aligned = []

    for conf, total_seg, i, j in cand_pairs:
        en_start, en_end, en_str, _ = en_candidates[i]
        ru_start, ru_end, ru_str, _ = ru_candidates[j]

        # Проверка на минимальное количество букв
        if len(re.findall(r'[a-zA-Zа-яА-ЯёЁ]', en_str)) < 3 or len(re.findall(r'[a-zA-Zа-яА-ЯёЁ]', ru_str)) < 3:
            continue

        # Проверка чисел
        en_nums = set(re.findall(r'\d+', en_str))
        ru_nums = set(re.findall(r'\d+', ru_str))
        if en_nums and ru_nums and not en_nums.intersection(ru_nums):
            continue

        # Проверка длины
        len_e = max(1, len(en_str))
        len_r = max(1, len(ru_str))
        ratio = len_r / len_e
        if ratio < 0.45 or ratio > 2.2:
            continue

        # Проверка непересечения с уже выбранными сегментами
        en_set = set(range(en_start, en_end))
        ru_set = set(range(ru_start, ru_end))
        if not (en_set & used_en) and not (ru_set & used_ru):
            len_diff = abs(len_e - len_r)
            aligned.append((en_start, en_end, ru_start, ru_end, en_str, ru_str, conf, len_diff, total_seg))
            used_en.update(en_set)
            used_ru.update(ru_set)

    # Сортировка по началу в en
    aligned.sort(key=lambda x: x[0])
    return aligned

def stage2_chunk_for_nllb(perfect_pairs: list[tuple], target_chars: int = 200) -> list[tuple]:
    """Склеивание выровненных пар в чанки."""
    if not perfect_pairs:
        return []
    final_chunks = []
    curr_en, curr_ru = "", ""
    for pair in perfect_pairs:
        en_sent, ru_sent = pair[0], pair[1]
        if curr_en and len(curr_en) + len(en_sent) + 1 > target_chars:
            if curr_en.strip() and curr_ru.strip():
                final_chunks.append((curr_en.strip(), curr_ru.strip()))
            curr_en, curr_ru = en_sent, ru_sent
        else:
            curr_en = (curr_en + " " + en_sent) if curr_en else en_sent
            curr_ru = (curr_ru + " " + ru_sent) if curr_ru else ru_sent
    if curr_en.strip() and curr_ru.strip():
        final_chunks.append((curr_en.strip(), curr_ru.strip()))
    return final_chunks

def save_logs(confidence_log: list, length_log: list):
    """Сохраняет логи уверенности и разницы длин в JSONL файлы"""

    # Лог уверенности (сортировка от неуверенного к уверенному)
    if confidence_log:
        confidence_log.sort(key=lambda x: x[2])  # Сортировка по confidence (возрастание)

        with open(CONFIDENCE_LOG, 'w', encoding='utf-8') as f:
            for en, ru, conf in confidence_log:
                record = {
                    "en": en,
                    "ru": ru,
                    "confidence": round(float(conf), 4)
                }
                f.write(json.dumps(record, ensure_ascii=False) + '\n')

        logger.info(f"Сохранён лог уверенности: {len(confidence_log)} записей в {CONFIDENCE_LOG}")

    # Лог разницы длин (сортировка от наибольшей разницы к наименьшей)
    if length_log:
        length_log.sort(key=lambda x: x[2], reverse=True)  # Сортировка по len_diff (убывание)

        with open(LENGTH_DIFF_LOG, 'w', encoding='utf-8') as f:
            for en, ru, diff in length_log:
                record = {
                    "en": en,
                    "ru": ru,
                    "len_diff": diff,
                    "en_len": len(en),
                    "ru_len": len(ru)
                }
                f.write(json.dumps(record, ensure_ascii=False) + '\n')

        logger.info(f"Сохранён лог разницы длин: {len(length_log)} записей в {LENGTH_DIFF_LOG}")

def main():
    """Основная функция обработки"""

    logger.info(f"Начало обработки файла: {INPUT_JSONL}")
    logger.info(f"Результаты будут сохранены в: {OUTPUT_JSONL}")

    # Статистика
    docs_processed = 0
    docs_with_pairs = 0
    total_chunks = 0
    total_pairs = 0

    # Для логов
    all_confidence = []
    all_length = []

    try:
        with open(INPUT_JSONL, 'r', encoding='utf-8') as f_in, \
             open(OUTPUT_JSONL, 'w', encoding='utf-8') as f_out:

            for line_num, line in enumerate(f_in, 1):
                if not line.strip():
                    continue

                try:
                    data = json.loads(line)
                except json.JSONDecodeError as e:
                    logger.warning(f"Строка {line_num}: ошибка парсинга JSON - {e}")
                    continue

                docs_processed += 1

                # Получаем тексты
                en_text = data.get("en", "")
                ru_text = data.get("ru", "")

                if not en_text or not ru_text:
                    logger.debug(f"Документ {docs_processed}: пустой EN или RU текст")
                    continue

                # --- Два прохода выравнивания ---
                # Проход 1: обычный сплит
                pairs_v1 = stage1_extended_filter(
                    en_text,
                    ru_text,
                    threshold=0.75,
                    max_merge=2,
                    split_commas=False
                )

                # Проход 2: с разбивкой по запятым
                pairs_v2 = stage1_extended_filter(
                    en_text,
                    ru_text,
                    threshold=0.75,
                    max_merge=2,
                    split_commas=True
                )

                # Объединение и жадный выбор непересекающихся
                all_pairs = pairs_v1 + pairs_v2

                # Сортировка: сначала по убыванию confidence, затем по возрастанию total_seg_count
                all_pairs.sort(key=lambda x: (-x[6], x[8]))

                used_en = set()
                used_ru = set()
                final_pairs = []

                for en_start, en_end, ru_start, ru_end, en_str, ru_str, conf, diff, total_seg in all_pairs:
                    en_set = set(range(en_start, en_end))
                    ru_set = set(range(ru_start, ru_end))
                    if not (en_set & used_en) and not (ru_set & used_ru):
                        final_pairs.append((en_str, ru_str, conf, diff))
                        used_en.update(en_set)
                        used_ru.update(ru_set)

                if final_pairs:
                    docs_with_pairs += 1
                    total_pairs += len(final_pairs)

                    # Собираем данные для логов
                    for en, ru, conf, diff in final_pairs:
                        all_confidence.append((en, ru, conf))
                        all_length.append((en, ru, diff))

                    # Этап 2: Создание чанков для NLLB
                    pairs_for_chunking = [(en, ru) for en, ru, _, _ in final_pairs]
                    nllb_chunks = stage2_chunk_for_nllb(pairs_for_chunking, target_chars=200)

                    # Сохраняем чанки
                    domain = data.get("domain", "")
                    doc_name = data.get("doc_name_en", data.get("doc_name", ""))

                    for chunk_en, chunk_ru in nllb_chunks:
                        row = {
                            "en": chunk_en,
                            "ru": chunk_ru,
                            "domain": domain,
                            "doc_name": doc_name
                        }
                        f_out.write(json.dumps(row, ensure_ascii=False) + '\n')
                        total_chunks += 1

                # Прогресс каждые 100 документов
                if docs_processed % 100 == 0:
                    logger.info(f"Обработано документов: {docs_processed}, "
                              f"найдено пар: {total_pairs}, "
                              f"создано чанков: {total_chunks}")

        logger.info(f"Обработка завершена!")
        logger.info(f"Всего обработано документов: {docs_processed}")
        logger.info(f"Документов с найденными парами: {docs_with_pairs}")
        logger.info(f"Всего найдено пар предложений: {total_pairs}")
        logger.info(f"Создано чанков для NLLB: {total_chunks}")

        # Сохраняем логи
        save_logs(all_confidence, all_length)

        # Дополнительная статистика
        if all_confidence:
            confidences = [conf for _, _, conf in all_confidence]
            logger.info(f"Статистика уверенности: min={min(confidences):.4f}, "
                      f"max={max(confidences):.4f}, "
                      f"avg={np.mean(confidences):.4f}")

        if all_length:
            diffs = [diff for _, _, diff in all_length]
            logger.info(f"Статистика разницы длин: min={min(diffs)}, "
                      f"max={max(diffs)}, "
                      f"avg={np.mean(diffs):.1f}")

    except FileNotFoundError:
        logger.error(f"Входной файл не найден: {INPUT_JSONL}")
    except Exception as e:
        logger.error(f"Неожиданная ошибка: {e}", exc_info=True)

if __name__ == "__main__":
    main()